In [1]:
import os
import json
import joblib
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

data_dir = '/content/drive/MyDrive/AML_Dataset/AML_Dataset'

print("Loading leakage-free datasets...")
train_df = pd.read_parquet(os.path.join(data_dir, 'train_features.parquet'))
test_df = pd.read_parquet(os.path.join(data_dir, 'test_features.parquet'))

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Mounted at /content/drive
Loading leakage-free datasets...
Train shape: (7189442, 20)
Test shape: (1784557, 20)


In [3]:
# Target and non-predictive metadata columns
target_col = 'Is Laundering'
drop_cols = ['Timestamp', 'Account', 'Account.1', target_col]

# Categorical columns that need encoding
categorical_cols = ['From Bank', 'To Bank', 'Payment Format', 'Payment Currency', 'Receiving Currency']

print("Encoding categorical variables...")
category_mappings = {}

for col in categorical_cols:
    if col in train_df.columns:
        train_df[col] = train_df[col].astype('category')
        test_df[col] = test_df[col].astype('category')

        # Save mapping dictionary for Person 4 inference
        category_mappings[col] = dict(enumerate(train_df[col].cat.categories))

# Separate features (X) and target (y)
X_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
y_test = test_df[target_col]

# Ensure exact feature column alignment
feature_names = list(X_train.columns)
X_test = X_test[feature_names]

Encoding categorical variables...


In [4]:
print("Training LightGBM baseline model...")

# Compute class imbalance ratio
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbose=-1
)

model.fit(X_train, y_train)
print("Model training complete.")

Training LightGBM baseline model...
Model training complete.


In [5]:
print("Evaluating model performance...")
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

pr_auc = average_precision_score(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("\n=== Model Metrics ===")
print(f"PR-AUC  (Primary Metric): {pr_auc:.4f}")
print(f"ROC-AUC (Secondary Metric): {roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Evaluating model performance...

=== Model Metrics ===
PR-AUC  (Primary Metric): 0.0083
ROC-AUC (Secondary Metric): 0.7921

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.79      0.88   1780762
           1       0.01      0.71      0.01      3795

    accuracy                           0.79   1784557
   macro avg       0.50      0.75      0.45   1784557
weighted avg       1.00      0.79      0.88   1784557



In [6]:
print("Evaluating model performance...")
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

pr_auc = average_precision_score(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("\n=== Model Metrics ===")
print(f"PR-AUC  (Primary Metric): {pr_auc:.4f}")
print(f"ROC-AUC (Secondary Metric): {roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Evaluating model performance...

=== Model Metrics ===
PR-AUC  (Primary Metric): 0.0083
ROC-AUC (Secondary Metric): 0.7921

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.79      0.88   1780762
           1       0.01      0.71      0.01      3795

    accuracy                           0.79   1784557
   macro avg       0.50      0.75      0.45   1784557
weighted avg       1.00      0.79      0.88   1784557

